In [3]:
import pandas as pd

# Leer el archivo parquet
df = pd.read_parquet('data/comercio_mexico.parquet')

# Ver las primeras 5 filas para confirmar
print(df.head())

  STATE COMMODITY                                               DESC  \
0     -    020130  MEAT OF BOVINE ANIMALS, BONELESS, FRESH OR CHI...   
1    AZ    020130  MEAT OF BOVINE ANIMALS, BONELESS, FRESH OR CHI...   
2    CA    020130  MEAT OF BOVINE ANIMALS, BONELESS, FRESH OR CHI...   
3    FL    020130  MEAT OF BOVINE ANIMALS, BONELESS, FRESH OR CHI...   
4    GA    020130  MEAT OF BOVINE ANIMALS, BONELESS, FRESH OR CHI...   

      VALOR month     flow  year  
0  93548947    01  imports  2025  
1   3445240    01  imports  2025  
2  53176050    01  imports  2025  
3   1743242    01  imports  2025  
4   3269897    01  imports  2025  


In [8]:
import pandas as pd

# Leer el archivo parquet
df = pd.read_parquet('data/ms_total.parquet')

# Ver las primeras 5 filas para confirmar
print(df.head())

  STATE COMMODITY            DESC     VALOR month     flow  year
0     -    010611  PRIMATES, LIVE  15911200    01  imports  2025
1    FL    010611  PRIMATES, LIVE     72000    01  imports  2025
2    MD    010611  PRIMATES, LIVE   7500000    01  imports  2025
3    PA    010611  PRIMATES, LIVE    419200    01  imports  2025
4    TX    010611  PRIMATES, LIVE   7920000    01  imports  2025


In [9]:
import pandas as pd

# Leer el archivo parquet
df = pd.read_parquet('data/xs_total.parquet')

# Ver las primeras 5 filas para confirmar
print(df.head())

  STATE COMMODITY                                    DESC      VALOR month  \
0     -    020319  MEAT OF SWINE, NESOI, FRESH OR CHILLED  114643450    01   
1    AL    020319  MEAT OF SWINE, NESOI, FRESH OR CHILLED      10000    01   
2    AR    020319  MEAT OF SWINE, NESOI, FRESH OR CHILLED    1070712    01   
3    AZ    020319  MEAT OF SWINE, NESOI, FRESH OR CHILLED     101330    01   
4    CA    020319  MEAT OF SWINE, NESOI, FRESH OR CHILLED   13799306    01   

      flow  year  
0  exports  2025  
1  exports  2025  
2  exports  2025  
3  exports  2025  
4  exports  2025  


In [5]:
import pandas as pd
import numpy as np

# 1. Leer el archivo original
df = pd.read_parquet('data/comercio_mexico.parquet')

# 2. LIMPIEZA: Asegurarnos de que 'VALOR' sea un número
# Si la columna es texto, le quitamos las comas y la convertimos a numérico
if df['VALOR'].dtype == 'object':
    df['VALOR'] = df['VALOR'].astype(str).str.replace(',', '', regex=False)
    # errors='coerce' convierte cualquier texto que no se pueda leer en un valor nulo (NaN)
    df['VALOR'] = pd.to_numeric(df['VALOR'], errors='coerce') 

# 3. Generar el multiplicador y realizar la operación
multiplicador = np.random.uniform(1, 10, size=len(df))
df['VALOR'] = df['VALOR'] * multiplicador

# 4. Guardar el resultado en un nuevo archivo parquet
df.to_parquet('data/comercio_total.parquet')

print("Archivo 'comercio_total.parquet' creado exitosamente y valores convertidos.")

Archivo 'comercio_total.parquet' creado exitosamente y valores convertidos.


In [7]:
# Filtrar por el estado de Alaska (AK) y asegurarnos de que el flujo sea 'imports'
df_alaska = df[(df['STATE'] == 'AK') & (df['flow'] == 'imports')]

# Ver los primeros resultados
df_alaska

,STATE,COMMODITY,DESC,VALOR,month,flow,year
6141,AK,730890,"STRUCTURES AND PARTS OF STRUCTURES NESOI, OF I...",4.812652e+04,01,imports,2025
6755,AK,730429,CASING AND TUBING OF A KIND USED IN DRILLING F...,1.261239e+06,01,imports,2025
9483,AK,847150,DIGITAL PROCESSING UNITS OTHER THAN THOSE OF 8...,2.200634e+04,01,imports,2025
10218,AK,854129,"TRANSISTORS, OTHER THAN PHOTOSENSITIVE, NESOI",8.322000e+03,01,imports,2025
12263,AK,841459,"FANS, NESOI",4.933316e+03,01,imports,2025
...,...,...,...,...,...,...,...
1284208,AK,847989,MACHINES AND MECHANICAL APPLIANCES HAVING INDI...,0.000000e+00,05,imports,2026
1285272,AK,902480,MACHINES AND APPLIANCES NESOI FOR TESTING THE ...,0.000000e+00,05,imports,2026
1285812,AK,842132,CATALYTIC CONVERTERS OR PARTICULATE FILTERS FO...,0.000000e+00,05,imports,2026
1285933,AK,821192,"KNIVES, OTHER THAN TABLE KNIVES, HAVING FIXED ...",0.000000e+00,05,imports,2026


In [1]:
import pandas as pd
import os

# 1. Rutas a los archivos (asumiendo que están en la carpeta 'data')
ruta_ms = os.path.join("data", "ms_total.parquet")
ruta_xs = os.path.join("data", "xs_total.parquet")

print("Cargando archivos individuales...")
df_ms = pd.read_parquet(ruta_ms)
df_xs = pd.read_parquet(ruta_xs)

print("Concatenando bases de datos...")
df_tot = pd.concat([df_ms, df_xs], ignore_index=True)

# 2. OPTIMIZACIÓN DE MEMORIA (Convierte textos repetitivos a categorías para bajar el peso del archivo)
columnas_categoricas = ['STATE', 'flow', 'month']
for col in columnas_categoricas:
    if col in df_tot.columns:
        df_tot[col] = df_tot[col].astype('category')

# 3. Guardar el archivo unificado
ruta_salida = os.path.join("data", "comercio_total.parquet")
print(f"Guardando archivo unificado optimizado en {ruta_salida}...")
df_tot.to_parquet(ruta_salida, index=False)

print("¡Proceso completado con éxito!")

Cargando archivos individuales...
Concatenando bases de datos...
Guardando archivo unificado optimizado en data\comercio_total.parquet...
¡Proceso completado con éxito!


In [2]:
import pandas as pd
import os

def limpiar_y_optimizar(df):
    # 1. Limpieza a numérico
    df['VALOR'] = pd.to_numeric(df['VALOR'].astype(str).str.replace(',', '').str.replace(' ', ''), errors='coerce').fillna(0)
    
    # 2. Formateo de fechas
    df['year'] = pd.to_numeric(df['year'], errors='coerce').fillna(0).astype(int)
    df['month'] = df['month'].astype(str).str.zfill(2)
    
    # 3. Crear capítulo
    df['Chapter'] = df['COMMODITY'].astype(str).str.zfill(6).str[:2]
    
    # 4. OPTIMIZACIÓN EXTREMA DE RAM: Convertir texto repetitivo a 'Category'
    columnas_categoricas = ['STATE', 'flow', 'month', 'Chapter']
    for col in columnas_categoricas:
        if col in df.columns:
            df[col] = df[col].astype('category')
            
    return df

print("Procesando datos de México...")
df_mex = pd.read_parquet(os.path.join("data", "comercio_mexico.parquet"))
df_mex_clean = limpiar_y_optimizar(df_mex)

print("Procesando datos Totales (MS y XS)...")
df_ms = pd.read_parquet(os.path.join("data", "ms_total.parquet"))
df_xs = pd.read_parquet(os.path.join("data", "xs_total.parquet"))
df_tot = pd.concat([df_ms, df_xs], ignore_index=True)
df_tot_clean = limpiar_y_optimizar(df_tot)

# Sobrescribimos los archivos originales con las versiones hiper-ligeras
df_mex_clean.to_parquet(os.path.join("data", "comercio_mexico.parquet"), index=False)
df_tot_clean.to_parquet(os.path.join("data", "comercio_total.parquet"), index=False)

print("¡Archivos listos para subir a GitHub!")

Procesando datos de México...
Procesando datos Totales (MS y XS)...
¡Archivos listos para subir a GitHub!


In [3]:
import pandas as pd
import os

# Rutas de entrada
ruta_mex_pq = os.path.join("data", "comercio_mexico.parquet")
ruta_tot_pq = os.path.join("data", "comercio_total.parquet")

# Rutas de salida
ruta_mex_csv = os.path.join("data", "comercio_mexico.csv")
ruta_tot_csv = os.path.join("data", "comercio_total.csv")

print("1/2 Convirtiendo comercio_mexico a CSV...")
df_mex = pd.read_parquet(ruta_mex_pq)
df_mex.to_csv(ruta_mex_csv, index=False, encoding='utf-8-sig')

print("2/2 Convirtiendo comercio_total a CSV...")
df_tot = pd.read_parquet(ruta_tot_pq)
df_tot.to_csv(ruta_tot_csv, index=False, encoding='utf-8-sig')

print("¡Archivos CSV generados exitosamente para uso local!")

1/2 Convirtiendo comercio_mexico a CSV...
2/2 Convirtiendo comercio_total a CSV...
¡Archivos CSV generados exitosamente para uso local!


In [4]:
import pandas as pd
import os

print("1. Cargando archivos originales...")
df_mex = pd.read_parquet(os.path.join("data", "comercio_mexico.parquet"))
df_ms = pd.read_parquet(os.path.join("data", "ms_total.parquet"))
df_xs = pd.read_parquet(os.path.join("data", "xs_total.parquet"))

# Unimos los totales
df_tot = pd.concat([df_ms, df_xs], ignore_index=True)

print("2. Creando diccionario maestro de descripciones...")
# Juntamos todos los commodities y descripciones de ambas bases para no perder ninguno
df_desc_all = pd.concat([df_mex[['COMMODITY', 'DESC']], df_tot[['COMMODITY', 'DESC']]])
# Eliminamos duplicados y nos quedamos con una tabla única
df_dict = df_desc_all.drop_duplicates(subset=['COMMODITY']).dropna(subset=['COMMODITY']).reset_index(drop=True)

# Guardamos el diccionario
df_dict.to_parquet(os.path.join("data", "diccionario_desc.parquet"), index=False)
print("   ✅ Diccionario guardado.")

print("3. Aplicando extirpación y compresión máxima (Downcasting)...")
def super_optimizar(df):
    # a) Extirpamos la descripción
    if 'DESC' in df.columns:
        df = df.drop(columns=['DESC'])
        
    # b) Limpieza del valor a numérico flotante
    df['VALOR'] = pd.to_numeric(df['VALOR'].astype(str).str.replace(',', '').str.replace(' ', ''), errors='coerce').fillna(0)
    
    # c) Aseguramos ceros a la izquierda en Commodity para que no se rompan
    df['COMMODITY'] = df['COMMODITY'].astype(str).str.zfill(6)
    
    # d) Creamos el Capítulo
    df['Chapter'] = df['COMMODITY'].str[:2]
    
    # e) DOWNCASTING: Convertimos a categorías y enteros pequeños
    df['year'] = pd.to_numeric(df['year'], errors='coerce').fillna(0).astype('int16')
    df['month'] = df['month'].astype(str).str.zfill(2).astype('category')
    df['STATE'] = df['STATE'].astype('category')
    df['flow'] = df['flow'].astype('category')
    df['Chapter'] = df['Chapter'].astype('category')
    
    # Para el commodity, convertirlo a categoría también ahorra mucha RAM
    df['COMMODITY'] = df['COMMODITY'].astype('category')
    
    return df

df_mex_opt = super_optimizar(df_mex)
df_tot_opt = super_optimizar(df_tot)

print("4. Guardando bases de datos ultra-ligeras...")
df_mex_opt.to_parquet(os.path.join("data", "comercio_mexico_opt.parquet"), index=False)
df_tot_opt.to_parquet(os.path.join("data", "comercio_total_opt.parquet"), index=False)

print("🎉 ¡Listo! Sube 'comercio_mexico_opt.parquet', 'comercio_total_opt.parquet' y 'diccionario_desc.parquet' a GitHub.")

1. Cargando archivos originales...
2. Creando diccionario maestro de descripciones...
   ✅ Diccionario guardado.
3. Aplicando extirpación y compresión máxima (Downcasting)...
4. Guardando bases de datos ultra-ligeras...
🎉 ¡Listo! Sube 'comercio_mexico_opt.parquet', 'comercio_total_opt.parquet' y 'diccionario_desc.parquet' a GitHub.


In [5]:
import pandas as pd
import os

# Capítulos de la Sección I (01-05) y Sección II (06-14)
capitulos_validos = [str(i).zfill(2) for i in range(1, 15)]

print("1. Cargando archivos optimizados actuales...")
ruta_mex = os.path.join("data", "comercio_mexico_opt.parquet")
ruta_tot = os.path.join("data", "comercio_total_opt.parquet")
ruta_dict = os.path.join("data", "diccionario_desc.parquet")

df_mex = pd.read_parquet(ruta_mex)
df_tot = pd.read_parquet(ruta_tot)
df_dict = pd.read_parquet(ruta_dict)

print("2. Extirpando todos los sectores ajenos a la Sección I y II...")
# Filtramos las bases principales
df_mex = df_mex[df_mex['Chapter'].isin(capitulos_validos)]
df_tot = df_tot[df_tot['Chapter'].isin(capitulos_validos)]

# Filtramos el diccionario (tomando los primeros 2 dígitos del COMMODITY)
df_dict = df_dict[df_dict['COMMODITY'].astype(str).str.zfill(6).str[:2].isin(capitulos_validos)]

print("3. Sobrescribiendo los archivos (Mismo nombre, fracción del tamaño)...")
df_mex.to_parquet(ruta_mex, index=False)
df_tot.to_parquet(ruta_tot, index=False)
df_dict.to_parquet(ruta_dict, index=False)

print("🎉 ¡Bases de datos reducidas con éxito! Listas para subir a GitHub.")

1. Cargando archivos optimizados actuales...
2. Extirpando todos los sectores ajenos a la Sección I y II...
3. Sobrescribiendo los archivos (Mismo nombre, fracción del tamaño)...
🎉 ¡Bases de datos reducidas con éxito! Listas para subir a GitHub.


In [1]:
import pandas as pd

# Leer el archivo parquet
df = pd.read_parquet('data/diccionario_desc.parquet')

# Ver las primeras 5 filas para confirmar
print(df.head())

  COMMODITY                                               DESC
0    020130  MEAT OF BOVINE ANIMALS, BONELESS, FRESH OR CHI...
1    020230           MEAT OF BOVINE ANIMALS, BONELESS, FROZEN
2    020319             MEAT OF SWINE, NESOI, FRESH OR CHILLED
3    020322  MEAT OF SWINE, HAMS, SHOULDERS AND CUTS THEREO...
4    020329                       MEAT OF SWINE, NESOI, FROZEN


In [2]:
import pandas as pd

# Define la ruta de tu diccionario (ajusta si es necesario)
ruta_archivo = 'data/diccionario_desc.parquet'

# 1. Cargar el DataFrame
df_dict = pd.read_parquet(ruta_archivo)

# 2. Castear explícitamente a string para evitar la pérdida de ceros a la izquierda en la nomenclatura
df_dict['COMMODITY'] = df_dict['COMMODITY'].astype(str)

# 3. Limpiar las descripciones:
# - Reemplaza los saltos de línea (\n) y retornos de carro (\r) por un espacio.
# - Reduce los múltiples espacios consecutivos a un solo espacio.
# - Elimina espacios residuales al inicio o al final (.str.strip()).
df_dict['DESC'] = df_dict['DESC'].replace(r'[\n\r]+', ' ', regex=True)
df_dict['DESC'] = df_dict['DESC'].replace(r'\s+', ' ', regex=True).str.strip()

# 4. Sobrescribir el archivo Parquet original
df_dict.to_parquet(ruta_archivo, index=False)
print("Archivo limpiado y reescrito con éxito.")

Archivo limpiado y reescrito con éxito.


In [3]:
import pandas as pd

# Cargar el archivo que ya procesaste
ruta_archivo = 'data/diccionario_desc.parquet'
df_dict = pd.read_parquet(ruta_archivo)

# Códigos HTS exactos que aparecen rotos en tu imagen
codigos_problema = ['020120', '030441', '020629', '051199']

# Filtramos para ver solo esos casos
df_test = df_dict[df_dict['COMMODITY'].isin(codigos_problema)]

for index, row in df_test.iterrows():
    print(f"=== HTS: {row['COMMODITY']} ===")
    
    # repr() expone los caracteres invisibles (mostrará \n, \t, \x0b, \u2028, etc.)
    print("Crudo (repr):", repr(row['DESC']))
    print()
    
    # Extra: Desglose byte por byte para ver si hay un carácter extraño rompiendo textwrap
    caracteres_raros = [c for c in row['DESC'] if not (32 <= ord(c) <= 126)]
    if caracteres_raros:
        print("¡ATENCIÓN! Se encontraron caracteres no-ASCII o de control:")
        for c in caracteres_raros:
            print(f" -> Carácter: {repr(c)}, Código Unicode: {hex(ord(c))}")
    else:
        print("El texto está limpio de caracteres extraños convencionales.")
        
    print("\n" + "="*50 + "\n")

=== HTS: 051199 ===
Crudo (repr): 'ANIMAL PRODUCTS, NESOI; DEAD HORSES AND OTHER EQUINE ANIMALS, BOVINE ANIMALS, SHEEP, GOATS AND POULTRY, UNFIT FOR HUMAN CONSUMPTION, NESOI'

El texto está limpio de caracteres extraños convencionales.


=== HTS: 020120 ===
Crudo (repr): 'MEAT OF BOVINE ANIMALS, CUTS WITH BONE IN (OTHER THAN HALF OR WHOLE CARCASSES), FRESH OR CHILLED'

El texto está limpio de caracteres extraños convencionales.


=== HTS: 020629 ===
Crudo (repr): 'OFFAL OF BOVINE ANIMALS, EDIBLE, NESOI, FROZEN'

El texto está limpio de caracteres extraños convencionales.


=== HTS: 030441 ===
Crudo (repr): 'PACIFIC SALMON (ONCORHYNCHUS NERKA, GORBUSCHA, KETA, ETC.), ATLANTIC SALMON (SALMO SALAR) AND DANUBE SALMON (HUCHO HUCHO) FILLETS, FRESH OR CHILLED'

El texto está limpio de caracteres extraños convencionales.


